# 04 - Feature Engineering & Machine Learning

I used past-hour and calendar features so the test period behaves more like a real forecast.

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

hourly = pd.read_csv("../01_Data/processed/hourly_energy.csv", parse_dates=["datetime"]).set_index("datetime")

In [ ]:
for lag in [1, 24, 168]:
    hourly[f"lag_{lag}"] = hourly["energy_kwh"].shift(lag)
for w in [24, 168]:
    hourly[f"rolling_{w}"] = hourly["energy_kwh"].shift(1).rolling(w).mean()

hourly["hour"] = hourly['date_time'].hour
hourly["dayofweek"] = hourly['date_time'].dayofweek
hourly["month"] = hourly['date_time'].month
hourly["is_weekend"] = (hourly["dayofweek"] >= 5).astype(int)
hourly = hourly.dropna()

In [ ]:
features = ["hour", "dayofweek", "month", "is_weekend",
            "lag_1", "lag_24", "lag_168", "rolling_24", "rolling_168"]
cut = int(len(hourly) * 0.8)
train = hourly.iloc[:cut]
test = hourly.iloc[cut:]

In [ ]:
models = {
    "Random Forest": RandomForestRegressor(n_estimators=120, max_depth=12, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=180, max_depth=2, learning_rate=0.04, loss="huber", random_state=42),
    "ANN": make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(32,16), max_iter=500, early_stopping=True, random_state=42))
}

for name, model in models.items():
    model.fit(train[features], train["energy_kwh"])
    pred = model.predict(test[features])
    print(name,
          "MAE:", round(mean_absolute_error(test["energy_kwh"], pred), 3),
          "RMSE:", round(mean_squared_error(test["energy_kwh"], pred)**0.5, 3),
          "R2:", round(r2_score(test["energy_kwh"], pred), 3))